# Notebook15 workflow definitions
Purpose: stable readout, ViT-equivalent geometry diagnostics, paired full-JEPA experiment and source-bound handoff. Inputs: pinned Food101 data, Notebook14 saved feature caches and new encoder/evaluator exports. Outputs: progress, plots, initial/final endpoints and checkpoints. Runtime: at most3hours in full profile; a few minutes for validation. Checkpoints: checkpoints/<run_id>/stages/<worker>/arms/. Exact resume: same RUN_ID and unchanged Notebook15 Run All before the persisted original deadline.

In [ ]:
"""Notebook15 source-bound staged research; imports define functions only."""
from pathlib import Path
import copy, hashlib, json, os, random, shlex, signal, statistics, subprocess, sys, time, types
import nbformat
import numpy as np
import torch
from fmm_jepa.training import atomic_json, digest_file
from fmm_jepa.food101_training import run_writer
from fmm_jepa.bootstrap import TaggedNotebookLoader
MAIN='notebooks/15_ViT_Equivalent_Geometry.ipynb'
SEEDS=[20261701,20261702,20261703,20261704]
VARIANTS=['vit','optional_geometry']
MODULES={}
def load(root,name,folder='15_vit_geometry'):
    path=Path(root)/'experiments'/folder/(name+'.ipynb');key=str(path.resolve())
    if key not in MODULES:
        m=types.ModuleType('fmm15_'+folder+'_'+name);m.__file__=str(path.resolve());TaggedNotebookLoader(path).exec_module(m);MODULES[key]=m
    return MODULES[key]
def read(path):return json.loads(Path(path).read_text())
def canonical(value):return json.loads(json.dumps(value,sort_keys=True,allow_nan=False))
def hash_object(value):return hashlib.sha256(json.dumps(canonical(value),sort_keys=True,separators=(',',':')).encode()).hexdigest()
def immutable(path,value):
    path=Path(path);value=canonical(value)
    if path.exists():
        if read(path)!=value:raise ValueError('Immutable record changed: '+str(path))
    else:atomic_json(path,value)
    return value
def consumer_hash(root):
    return hash_object([c.source for c in nbformat.read(Path(root)/MAIN,as_version=4).cells if c.cell_type=='code'])
def safe_id(value):
    if not value or any(not(c.isalnum()or c in '-_')for c in value):raise ValueError('Safe run ID required')
    return value
def study_configuration(run_id='vit-geometry-001',profile='full'):
    safe_id(run_id)
    if profile not in ('full','validation'):raise ValueError('Profile must be full or validation')
    return {'schema':'notebook15-study/v1','run_id':run_id,'profile':profile,'hard_limit_seconds':10800,
      'seeds':SEEDS,'variants':VARIANTS,'devices':['cuda:0','cuda:1'],'updates':600,'image_size':128,
      'effective_batch':128,'round_updates':50,'evaluation_reserve_seconds':1800,'probe_endpoints':['initial','final'],
      'probe_convex':{'l2':1e-4,'max_steps':1000,'gradient_inf_tolerance':1e-5,'relative_objective_tolerance':1e-9,'stable_checks':3},
      'quality_gate':{'minimum_mean_delta_pp':-.25,'minimum_nonnegative_pairs':3,'minimum_mean_initialization_gain_pp':.25,
        'maximum_mean_knn_regression_pp':.5},
      'candidate':'raw64D Q/K; signed optional spatial bias only learns; independent temperature and spatial length fixed',
      'cached14_readout':'All16 final checkpoints; converged train-objective readout and kNN20; developmental reanalysis',
      'official_test':'NOT_RUN','automatic_notebook16':False}
def base_config(root,configuration=None):
    conf=configuration or study_configuration();cfg=copy.deepcopy(read(Path(root)/'artifacts/notebook_14/routing-capacity-001/plan.json')['arms'][0]['config'])
    cfg.pop('kernel14',None);cfg['kernel15']={'variant':'vit','spatial_coefficient_bound':2.,'spatial_length':.35,
      'temperature_trainable':False,'spatial_length_trainable':False,'spatial_coefficient_trainable':True}
    cfg.update(numerical_backend='vit-equivalent-optional-signed-spatial-bias/v1',comparison=conf['candidate'])
    cfg['training'].update(updates=conf['updates'],checkpoint_every=50,microbatch=128,accumulation_steps=1)
    cfg['evaluation'].update(diagnostic_steps=[0,conf['updates']],heldout_enabled=False)
    cfg['budget'].update(max_seconds=conf['hard_limit_seconds'],target_seconds=conf['hard_limit_seconds'],
      evaluation_reserve_seconds=conf['evaluation_reserve_seconds'])
    cfg['probe_convex']=copy.deepcopy(conf['probe_convex']);cfg['probe_convex']['num_classes']=101
    cfg['probe'].update(epochs=20,selection='legacy_field_only_actual_converged_convex_probe')
    cfg['cpu_threads']=2;cfg['dtype']='float32';return cfg
def prepare_bundle(root,artifact,config,verify_images=True):
    root,artifact=Path(root).resolve(),Path(artifact).resolve()
    prep=load(root,'preparation','05_scaling')
    def paths(source_root,folder,stage):
        source_root,folder=Path(source_root).resolve(),Path(folder).resolve()
        rel=folder.relative_to(source_root/'artifacts/notebook_15');run_id=safe_id(rel.parts[0])
        return folder,source_root/'checkpoints'/run_id/'stages'/stage['id']/'calibration',run_id
    prep._stage_paths=paths
    return prep.prepare_stage_bundle(root,artifact,{'id':'learning','image_size':128,'pretrain_pool':'selected'},config,verify_images=verify_images)
def configure(config):
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8');torch.set_num_threads(int(config.get('cpu_threads',2)))
    torch.use_deterministic_algorithms(True);torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    if str(config['device']).startswith('cuda'):torch.cuda.set_device(torch.device(config['device']))
def stage_plan(root,configuration,bundle,source_hashes=None):
    base=base_config(root,configuration);arms=[]
    for i,seed in enumerate(configuration['seeds']):
        worker=i%len(configuration['devices'])
        for variant in configuration['variants']:
            cfg=copy.deepcopy(base);cfg['device']=configuration['devices'][worker];cfg['kernel15']['variant']=variant
            cfg['seeds'].update(model_init=seed,data_order=984101+1000*i,augmentation=985202+1000*i,mask=986303+1000*i,probe=4404)
            cfg['model']['feature_seed']=seed+1000
            arms.append({'id':'s'+str(seed)+'-'+variant,'kind':'vit'if variant=='vit'else'semantic','variant':variant,
              'condition':variant,'seed':seed,'worker':worker,'config':cfg})
    return {'schema':'notebook15-paired-plan/v1','run_id':configuration['run_id'],'stage_id':'learning','stage_role':'learning',
      'stage':{'id':'learning','image_size':128,'pretrain_pool':'selected'},'updates':configuration['updates'],
      'milestones':[configuration['updates']],'round_updates':configuration['round_updates'],
      'probe_endpoints':['initial','final'],'probe_convex':copy.deepcopy(configuration['probe_convex']),'hard_limit_seconds':configuration['hard_limit_seconds'],'effective_batch':128,
      'seeds':configuration['seeds'],'registered_seeds':configuration['seeds'],'variants':configuration['variants'],'arms':arms,
      'train_count':len(bundle['train_ids']),'probe_train_count':len(bundle['probe_train_ids']),
      'source_hashes':source_hashes or load(root,'runner').stage_definition_hashes(root),
      'consumer_notebook_sha256':consumer_hash(root),'resume_notebook':MAIN,'resume_cell':'launch-vit-geometry',
      'evaluation_reserve_seconds':configuration['evaluation_reserve_seconds'],'heldout_status':'NOT_RUN',
      'selection':'One optional-spatial candidate fixed before training; all paired seeds; train-objective probe convergence',
      'quality_gate':configuration['quality_gate'],'failure_policy':'First failure stops; cutoff/nonconverged endpoint UNKNOWN'}
def bootstrap_source(root):
    return next(c.source for c in nbformat.read(Path(root)/MAIN,as_version=4).cells if c.get('id')=='bootstrap-vit-geometry')
def session_exists(name):return subprocess.run(['tmux','has-session','-t','='+name],capture_output=True).returncode==0
def spawn(root,artifact,session,expression,path):
    root,artifact,path=Path(root),Path(artifact),Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    code=bootstrap_source(root)+'\n'+expression+'\n'
    if path.exists():
        if [c.source for c in nbformat.read(path,as_version=4).cells if c.cell_type=='code']!=[code]:raise ValueError('Worker notebook changed')
    else:
        nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Notebook15 detached execution\nPurpose: execute the source-bound registered phase. Inputs: immutable binding, original data and checkpoints. Outputs: resumable checkpoints and stage receipts. Runtime: remaining original three-hour clock. Exact resume: rerun unchanged Notebook15 with the same run ID before the deadline.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'name':'ahr-go','display_name':'AHR Go (.venv)','language':'python'}})
        nbformat.write(nb,path)
    if session_exists(session):return
    cmd=[sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',str(path)]
    shell='exec '+shlex.join(cmd)+' > '+shlex.quote(str(path.with_suffix('.log')))+' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),'bash','-c',shell],check=True)
def terminate_session_tree(session):
    if not session_exists(session):return []
    roots={int(x)for x in subprocess.check_output(['tmux','list-panes','-t','='+session,'-F','#{pane_pid}'],text=True).split()}
    edges=[tuple(map(int,line.split()))for line in subprocess.check_output(['ps','-eo','pid=,ppid='],text=True).splitlines()];owned=set(roots)
    while True:
        expanded=owned|{pid for pid,parent in edges if parent in owned}
        if expanded==owned:break
        owned=expanded
    for pid in sorted(owned-roots,reverse=True)+sorted(roots):
        try:os.kill(pid,signal.SIGKILL)
        except ProcessLookupError:pass
    subprocess.run(['tmux','kill-session','-t','='+session],capture_output=True);return sorted(owned)
def archive_attempt(path):
    path=Path(path)
    if path.exists():
        target=path.parent/'attempt_history'/(path.stem+'-'+str(time.time_ns())+path.suffix);target.parent.mkdir(parents=True,exist_ok=True);path.rename(target)
        return str(target)
def set_status(artifact,stage,status,**details):
    record={'schema':'notebook15-status/v1','stage':stage,'status':status,'updated_unix':time.time(),**details}
    atomic_json(Path(artifact)/'study_status.json',record);return record
def verify_binding(root,artifact,binding):
    if binding['source_hashes']!=load(root,'runner').stage_definition_hashes(root):raise ValueError('Scientific source changed')
    if binding['consumer_hash']!=consumer_hash(root):raise ValueError('Main notebook code changed')
    if binding['reference_plan_sha256']!=digest_file(Path(root)/'artifacts/notebook_14/routing-capacity-001/plan.json'):raise ValueError('Frozen referenceconfiguration changed')
    if read(Path(artifact)/'budget_clock.json')!=binding['clock']:raise ValueError('Original clock changed')
    return True
def readiness(root,configuration=None):
    root=Path(root);cfg=configuration or study_configuration();missing=[]
    for path in ['artifacts/notebook_14/routing-capacity-001/visual_report/analysis_results.json',
      'artifacts/notebook_02/food101-pilot-001/split_manifest.json','experiments/15_vit_geometry/encoder.ipynb',
      'experiments/15_vit_geometry/runner.ipynb','experiments/15_vit_geometry/evaluation.ipynb']:
        if not(root/path).exists():missing.append(path)
    if missing:raise FileNotFoundError('Required inputs: '+repr(missing))
    if cfg['profile']=='full'and torch.cuda.device_count()<len(cfg['devices']):raise RuntimeError('Two configured CUDA GPUs required; set devices before first launch for a separately timed protocol')
    if cfg['profile']=='full'and subprocess.run(['tmux','-V'],capture_output=True).returncode:raise RuntimeError('tmux unavailable')
    storage=subprocess.run(['findmnt','-T',str(root),'-n','-o','SOURCE,FSTYPE,TARGET'],capture_output=True,text=True).stdout.strip()
    return {'status':'READY','profile':cfg['profile'],'storage':storage,'persistent_boot_disk':storage.startswith('/dev/sda'),
      'instance_deletion_policy':'UNVERIFIED','official_test':'NOT_RUN','expected_runtime':'Measured by launch preflight; maximum3h'}
def launch_study(root,configuration=None):
    root=Path(root).resolve();conf=canonical(configuration or study_configuration());artifact=root/'artifacts/notebook_15'/safe_id(conf['run_id'])
    if conf['profile']=='validation':return validation_run(root,artifact)
    ready=readiness(root,conf)
    with run_writer(artifact/'launch_writer'):
        path=artifact/'study_binding.json'
        if path.exists():
            binding=read(path);verify_binding(root,artifact,binding)
            if binding['configuration']!=conf:raise ValueError('Configuration changed on resume')
            prior=read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
            if prior.get('status')in('COMPLETE','NO_GO'):return prior
            if time.time()>=binding['clock']['deadline_unix']:raise TimeoutError('Original three-hour clock expired; use saved report, never reset this run')
        else:
            if (artifact/'budget_clock.json').exists()or(artifact/'plan.json').exists():raise ValueError('Unbound existing artifacts')
            now=time.time();clock={'started_unix':now,'deadline_unix':now+conf['hard_limit_seconds'],'max_seconds':conf['hard_limit_seconds']}
            binding={'schema':'notebook15-binding/v1','configuration':conf,'source_hashes':load(root,'runner').stage_definition_hashes(root),
              'consumer_hash':consumer_hash(root),'clock':clock,'readiness':ready,'reference_plan_sha256':digest_file(root/'artifacts/notebook_14/routing-capacity-001/plan.json')}
            immutable(path,binding);immutable(artifact/'budget_clock.json',clock)
        main='fmm15-'+conf['run_id'];watch=main+'-deadline'
        if(artifact/'study_status.json').exists()and read(artifact/'study_status.json').get('status')in('FAILED','UNKNOWN'):archive_attempt(artifact/'study_status.json')
        set_status(artifact,'launch_or_exact_resume','RUNNING',deadline_unix=binding['clock']['deadline_unix'])
        spawn(root,artifact,watch,'WORKFLOW.run_watchdog(ROOT, '+repr(conf['run_id'])+')',artifact/'deadline_watchdog.ipynb')
        spawn(root,artifact,main,'WORKFLOW.run_study(ROOT, '+repr(conf['run_id'])+')',artifact/'coordinator.ipynb')
        record={'status':'LAUNCHED','session':main,'worker_sessions':[main+'-worker0',main+'-worker1'],'artifact':str(artifact),
          'deadline_unix':binding['clock']['deadline_unix'],'attach':'tmux attach -t '+main,'detach':'Ctrl-b then d',
          'resume':'Same RUN_ID, unchanged sources, Restart Kernel and Run All in '+MAIN+' before originaldeadline'}
        atomic_json(artifact/'launch.json',record);return record
def deadline_guard(deadline,margin=0):
    if time.time()>=deadline-margin:raise TimeoutError('Registered execution allowance exhausted; required incomplete results remain UNKNOWN')
def wait_workers(root,artifact,binding,jobs):
    while True:
        deadline_guard(binding['clock']['deadline_unix']);verify_binding(root,artifact,binding)
        results=[];progress=[]
        for job in jobs:
            p=Path(job['result']);folder=p.parent
            if p.exists():
                record=read(p)
                if record.get('status')!='COMPLETE':raise RuntimeError('Worker failed: '+repr(record))
                results.append(record)
            else:
                status=read(folder/'study_status.json')if(folder/'study_status.json').exists()else{}
                progress.append({'session':job['session'],**status})
                if status.get('status')in('FAILED','UNKNOWN'):raise RuntimeError('Worker terminal: '+repr(status))
                if not session_exists(job['session']):raise RuntimeError('Worker exited without a completion receipt: '+job['session'])
        atomic_json(Path(artifact)/'heartbeat.json',{'updated_unix':time.time(),'completed_workers':len(results),'progress':progress,
          'deadline_unix':binding['clock']['deadline_unix']})
        if len(results)==len(jobs):return results
        time.sleep(5)
def geometry_study(root,artifact,bundle):
    artifact=Path(artifact);out=artifact/'geometry_diagnostic.json';backend=load(root,'encoder');runner=load(root,'runner')
    from fmm_jepa.food101_data import EvaluationDataset
    source=read(Path(root)/'artifacts/notebook_14/routing-capacity-001/stage_evaluation.json')
    prior=source['evaluations']['s20261601-vit-u600'];record=prior['source_encoder']
    if digest_file(record['path'])!=record['sha256']:raise ValueError('Teacher checkpoint checksum changed')
    cfg=base_config(root);cfg['device']='cpu';ids=random.Random(20269915).sample(list(bundle['train_ids']),4)
    identity={'source_encoder':record,'training_ids':ids,'data_fingerprint':bundle['dataset'].fingerprint,
      'source_hashes':runner.stage_definition_hashes(root),'selection_seed':20269915,'method':'Descriptive one-constraint routing approximation on fourfixedrandomTRAINimages, all4blocks; no dev selection'}
    if out.exists():
        saved=read(out)
        if saved['identity']!=canonical(identity):raise ValueError('Geometry diagnostic identity changed')
        return saved
    model=runner.build_model(cfg,'vit').cpu().eval()
    state=torch.load(record['path'],map_location='cpu',weights_only=False)
    if 'model' in state:
        target={k[len('target.'):]:v for k,v in state['model'].items()if k.startswith('target.')}
    else:target=state
    model.target.load_state_dict(target)
    data=EvaluationDataset(bundle['dataset'],ids,cfg);images=torch.stack([data[i]['image']for i in range(len(ids))])
    rows=[]
    with torch.no_grad():
        for layer in range(int(cfg['model']['depth'])):
            states=backend.frozen_attention_states(model.target,images,layer_index=layer)
            for row in backend.diagnostic_geometry(states['q'],states['k'],states['v'],states['valid'],states['positions']):
                rows.append({'layer':layer,**row})
    result={'status':'COMPLETE','identity':canonical(identity),'rows':rows,'quality_evidence':False,
      'selection':'optional_geometry fixed before this diagnostic; errors isolate constraints, not classification quality'}
    immutable(out,result);return result
def run_worker(root,run_id,worker):
    root=Path(root);artifact=root/'artifacts/notebook_15'/safe_id(run_id);binding=read(artifact/'study_binding.json');verify_binding(root,artifact,binding)
    folder=artifact/'workers'/('worker'+str(worker));plan=read(folder/'plan.json');runner,evaluation=load(root,'runner'),load(root,'evaluation')
    cfg=plan['arms'][0]['config'];configure(cfg);deadline=binding['clock']['deadline_unix']
    bundle=prepare_bundle(root,folder/'data',cfg,verify_images=True)
    with run_writer(folder/'worker_writer'):
        try:
            set_status(folder,'actual_configuration_resume_preflight','RUNNING');runner.prepare_stage(root,folder,plan,bundle)
            preflight=folder/'preflight_result.json'
            atomic_json(preflight,runner.verify_stage_resume(root,folder,plan,bundle,deadline))
            if read(preflight).get('status')!='PASS':raise RuntimeError('Actual-config full-state exactresume failed')
            probe_check=folder/'probe_preflight.json'
            atomic_json(probe_check,evaluation.verify_evaluation_smoke(root,folder,plan,bundle,deadline))
            if read(probe_check).get('status')!='PASS':raise RuntimeError('Convex-probe exactresume failed')
            deadline_guard(deadline,plan['evaluation_reserve_seconds']);set_status(folder,'paired_full_jepa_training','RUNNING',updates=plan['updates'])
            tp=folder/'training_result.json'
            if tp.exists()and read(tp).get('status')=='COMPLETE':training=read(tp)
            else:training=runner.train_stage(root,folder,plan,bundle,deadline);atomic_json(tp,training)
            set_status(folder,'initial_and_final_converged_readouts','RUNNING')
            result=evaluation.evaluate_stage(root,folder,plan,bundle,training,deadline)
            if result.get('status')!='COMPLETE':raise TimeoutError('Required converged endpoints incomplete: '+str(result.get('status')))
            verify_binding(root,artifact,binding);record={'status':'COMPLETE','worker':worker,'completed_models':len(plan['arms']),
              'training_result':str(tp),'evaluation_result':str(folder/'stage_evaluation.json')}
            atomic_json(folder/'worker_result.json',record);set_status(folder,'complete','COMPLETE');return record
        except BaseException as e:
            record={'status':'UNKNOWN'if isinstance(e,TimeoutError)else'FAILED','error':repr(e),'worker':worker}
            atomic_json(folder/'worker_result.json',record);set_status(folder,'interrupted',record['status'],error=repr(e));raise
def _endpoint_rows(evaluation):
    entries=evaluation.get('evaluations',{})
    return list(entries.values())if isinstance(entries,dict)else list(entries)
def screen(plan,evaluation):
    rows=_endpoint_rows(evaluation);seeds=plan['seeds'];effects=[];gains=[];baseline_gains=[];knn=[]
    for seed in seeds:
        def row(variant,endpoint):
            matches=[r for r in rows if int(r['seed'])==seed and r['variant']==variant and r['endpoint']==endpoint]
            if len(matches)!=1:raise ValueError('Missing/duplicate registered endpoint')
            r=matches[0]
            if not r.get('converged')or not r.get('numerical_valid',False):raise TimeoutError('Nonconverged/invalid endpoint')
            return r
        v,c,i=row('vit','final'),row('optional_geometry','final'),row('optional_geometry','initial');vi=row('vit','initial')
        effects.append(c['accuracy_percent']-v['accuracy_percent']);gains.append(c['accuracy_percent']-i['accuracy_percent']);baseline_gains.append(v['accuracy_percent']-vi['accuracy_percent'])
        knn.append(c['knn_accuracy_percent']-v['knn_accuracy_percent'])
    gate=plan['quality_gate'];mean=float(np.mean(effects));gain=float(np.mean(gains));baseline_gain=float(np.mean(baseline_gains));kg=float(np.mean(knn))
    passed=mean>=gate['minimum_mean_delta_pp']and sum(d>=0 for d in effects)>=gate['minimum_nonnegative_pairs']and gain>=gate['minimum_mean_initialization_gain_pp']and baseline_gain>=gate['minimum_mean_initialization_gain_pp']and kg>=-gate['maximum_mean_knn_regression_pp']
    from scipy import stats
    ci=stats.t.interval(.95,len(effects)-1,loc=mean,scale=stats.sem(effects))if len(effects)>1 and np.std(effects,ddof=1)>0 else None
    return {'status':'PROMISING_FOR_COMPRESSION'if passed else'NO_GO_UNCHANGED_GEOMETRY','quality_pass':bool(passed),
      'n_pairs':len(seeds),'mean_delta_pp':mean,'paired_seed_deltas_pp':effects,'positive_pairs':sum(d>0 for d in effects),
      'mean_own_initialization_gain_pp':gain,'mean_vit_initialization_gain_pp':baseline_gain,'paired_initialization_gains_pp':gains,'mean_knn_delta_pp':kg,
      'raw_ci95_pp':[float(x)for x in ci]if ci is not None else None,'role':'Four-pair reused-development screening; not confirmatory superiority'}
def make_report(root,artifact,plan,evaluation,diagnostic=None,cached=None):
    import matplotlib;matplotlib.use('Agg');import matplotlib.pyplot as plt
    root,artifact=Path(root),Path(artifact);report=artifact/'visual_report';report.mkdir(parents=True,exist_ok=True)
    outcome=screen(plan,evaluation);rows=_endpoint_rows(evaluation);title_prefix='Software fixture — 'if evaluation.get('fixture_source')else''
    colors={'vit':'#315f8c','optional_geometry':'#c27228'};fig,ax=plt.subplots(figsize=(9,5))
    for j,variant in enumerate(plan['variants']):
        for offset,endpoint,marker in [(-.12,'initial','o'),(.12,'final','D')]:
            values=[r['accuracy_percent']for r in rows if r['variant']==variant and r['endpoint']==endpoint]
            ax.scatter([j+offset]*len(values),values,color=colors[variant],marker=marker,facecolors='none'if endpoint=='initial'else colors[variant],label=variant+' '+endpoint)
    ax.set_xticks(range(2),['ViT','Optional spatial geometry']);ax.set_ylim(bottom=0);ax.set_ylabel('Development top-1 accuracy (%)')
    ax.set_title(title_prefix+'Converged linear readout — initial and final JEPA features');ax.legend(loc='best');ax.grid(axis='y',alpha=.2);fig.tight_layout();fig.savefig(report/'quality.png',dpi=150);plt.close(fig)
    fig,ax=plt.subplots(figsize=(9,4));effects=outcome['paired_seed_deltas_pp'];ax.scatter(range(1,len(effects)+1),effects,color=colors['optional_geometry'],s=70)
    ax.axhline(0,color='black',linewidth=1);ax.axhline(-.25,color='#8a7640',linestyle='--',label='−0.25 pp screening margin')
    ax.set_xticks(range(1,len(effects)+1));ax.set_xlabel('Paired seed index (independent replicates)');ax.set_ylabel('Candidate − ViT (percentage points)');ax.set_title(title_prefix+'Every registered final paired effect');ax.legend();fig.tight_layout();fig.savefig(report/'paired_effects.png',dpi=150);plt.close(fig)
    if diagnostic and diagnostic.get('rows'):
        groups={}
        for row in diagnostic['rows']:
            key=row.get('constraint',row.get('case',row.get('variant',row.get('name','case'))));value=row.get('relative_output_l2_error',row.get('relative_output_error',row.get('output_relative_error',row.get('relative_error'))))
            if value is not None:groups.setdefault(key,[]).append(float(value))
        if groups:
            fig,ax=plt.subplots(figsize=(10,5));names=list(groups);ax.barh(names,[np.mean(groups[n])for n in names],color='#315f8c');ax.set_xlabel('Relative attention-output error against frozen ViT');ax.set_title('One constraint at a time — four training images, four blocks');fig.tight_layout();fig.savefig(report/'geometry_errors.png',dpi=150);plt.close(fig)
    if cached and cached.get('readout_stability_comparison'):
        summary=cached['readout_stability_comparison'];names=[row['variant'] for row in summary];indices=np.arange(len(names))
        fig,ax=plt.subplots(figsize=(11,5))
        for offset,key,label,color in [(-.25,'legacy_epoch20_mean_accuracy_percent','Registered epoch20','#8a7640'),(0.,'converged_mean_accuracy_percent','Converged linear probe','#315f8c'),(.25,'knn_mean_accuracy_percent','Cosine kNN20','#c27228')]:
            ax.bar(indices+offset,[row[key] for row in summary],width=.24,label=label,color=color)
        ax.set_xticks(indices,[name.replace('_','\n') for name in names]);ax.set_ylabel('Development accuracy (%)');ax.set_title('Same16 saved encoders — readout comparison; no new JEPA training');ax.set_ylim(bottom=0);ax.legend();fig.tight_layout();fig.savefig(report/'readout_stability.png',dpi=150);plt.close(fig)
    result={'status':'COMPLETE','fixture_source':evaluation.get('fixture_source'),'run_id':plan['run_id'],'screening':outcome,'official_test':'NOT_RUN','no_notebook16_launched':True,
      'evaluation':str(artifact/'stage_evaluation.json'),'figures':[str(p)for p in sorted(report.glob('*.png'))],
      'limitations':['Reused development images; four pairs are screening only.','Candidate changes only optional spatial coefficient in this registered training screen.','Converged readout differs from Notebook14 fixed20 protocol; absolute reanalysis scores are not new pretraining results.','Dense candidate has no FMM speedup; compression and optimized full-JEPA scaling belong to Notebook16.']}
    atomic_json(report/'results.json',result)
    handoff={'schema':'notebook15-to16/v1','status':'COMPLETE','quality_pass':outcome['quality_pass'],'run_id':plan['run_id'],
      'candidate_variant':'optional_geometry','candidate_plan':str(artifact/'plan.json'),'plan_sha256':digest_file(artifact/'plan.json'),
      'training_result':str(artifact/'training_result.json'),'training_sha256':digest_file(artifact/'training_result.json'),
      'evaluation_result':str(artifact/'stage_evaluation.json'),'evaluation_sha256':digest_file(artifact/'stage_evaluation.json'),
      'source_hashes':plan['source_hashes'],'screening':outcome,'official_test':'NOT_RUN','candidate_checkpoint_records':{},
      'data':str(artifact/'data'),'token_source_role':'Training-only frozen EMA states; every seed retained'}
    training=read(artifact/'training_result.json')
    for arm in plan['arms']:
        handoff['candidate_checkpoint_records'][arm['id']]=training['arms'][arm['id']]['milestones'][str(plan['updates'])]
    atomic_json(artifact/'handoff_15_to_16.json',handoff);return result
def run_study(root,run_id):
    root=Path(root);artifact=root/'artifacts/notebook_15'/safe_id(run_id);binding=read(artifact/'study_binding.json');verify_binding(root,artifact,binding)
    with run_writer(artifact/'coordinator_writer'):
        jobs=[]
        try:
            conf=binding['configuration'];deadline=binding['clock']['deadline_unix'];base=base_config(root,conf);configure(base)
            set_status(artifact,'verified_data','RUNNING');bundle=prepare_bundle(root,artifact/'data',base,verify_images=True)
            plan=stage_plan(root,conf,bundle,binding['source_hashes']);immutable(artifact/'plan.json',plan)
            set_status(artifact,'frozen_teacher_constraint_diagnostic','RUNNING');diagnostic=geometry_study(root,artifact,bundle)
            set_status(artifact,'cached14_converged_readout_reanalysis','RUNNING')
            cached=load(root,'evaluation').cached14_study(root,artifact,deadline_unix=deadline-conf['evaluation_reserve_seconds'])
            if cached.get('status')!='COMPLETE':raise TimeoutError('Cached readout stability study did not converge within allowance')
            for worker in range(len(conf['devices'])):
                folder=artifact/'workers'/('worker'+str(worker));folder.mkdir(parents=True,exist_ok=True)
                child=copy.deepcopy(plan);child['stage_id']='worker'+str(worker);child['seeds']=sorted({a['seed']for a in plan['arms']if a['worker']==worker});child['arms']=[a for a in plan['arms']if a['worker']==worker]
                child['parent_plan_hash']=hash_object(plan);immutable(folder/'plan.json',child)
                session='fmm15-'+run_id+'-worker'+str(worker);p=folder/'worker_result.json'
                if p.exists()and read(p).get('status')!='COMPLETE':
                    archive_attempt(p);archive_attempt(folder/'study_status.json')
                if not p.exists():
                    set_status(folder,'exact_resume_pending','RUNNING')
                    spawn(root,artifact,session,'WORKFLOW.run_worker(ROOT, '+repr(run_id)+', '+str(worker)+')',folder/'worker.ipynb')
                jobs.append({'session':session,'result':str(p)})
            set_status(artifact,'paired_training_and_converged_evaluation','RUNNING');results=wait_workers(root,artifact,binding,jobs)
            evaluations={};arms={}
            for item in results:
                e=read(item['evaluation_result']);t=read(item['training_result']);evaluations.update(e['evaluations']);arms.update(t['arms'])
            if len(arms)!=8 or len(evaluations)!=16:raise ValueError('All8models and16initial/final endpoints required')
            evaluation={'status':'COMPLETE','evaluations':evaluations,'run_id':run_id,'official_test':'NOT_RUN'}
            training={'status':'COMPLETE','arms':arms,'planned_updates':plan['updates'],'equal_updates':True,'paired_traces_within_seed':True,'no_failed_attempts':True}
            atomic_json(artifact/'stage_evaluation.json',evaluation);atomic_json(artifact/'training_result.json',training)
            result=make_report(root,artifact,plan,evaluation,diagnostic,cached);verify_binding(root,artifact,binding)
            return set_status(artifact,'complete','COMPLETE',elapsed_seconds=time.time()-binding['clock']['started_unix'],report=result,
              next_stage='Notebook16 may be run separately; no automatic long-run launch')
        except BaseException as e:
            for job in jobs:
                if session_exists(job['session']):terminate_session_tree(job['session'])
            set_status(artifact,'interrupted','UNKNOWN'if isinstance(e,TimeoutError)else'FAILED',error=repr(e),conclusion='Required incomplete endpoints remainUNKNOWN');raise
def run_watchdog(root,run_id):
    root=Path(root);artifact=root/'artifacts/notebook_15'/safe_id(run_id);binding=read(artifact/'study_binding.json');main='fmm15-'+run_id
    while True:
        try:verify_binding(root,artifact,binding)
        except BaseException as error:
            killed={name:terminate_session_tree(name)for name in [main,main+'-worker0',main+'-worker1']}
            return set_status(artifact,'binding_guard','FAILED',error=repr(error),terminated_pids=killed)
        status=read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
        if status.get('status')in('COMPLETE','NO_GO','FAILED','UNKNOWN'):
            for name in [main,main+'-worker0',main+'-worker1']:
                if session_exists(name):terminate_session_tree(name)
            return {'status':'TERMINAL','study_status':status.get('status')}
        if time.time()>=binding['clock']['deadline_unix']:
            killed={name:terminate_session_tree(name)for name in [main,main+'-worker0',main+'-worker1']}
            record=set_status(artifact,'absolute_deadline','UNKNOWN',reason='Original three-hour cap enforced; checkpoints retained',terminated_pids=killed)
            atomic_json(artifact/'watchdog_result.json',record);return record
        time.sleep(5)
def inspect_study(root,run_id='vit-geometry-001'):
    artifact=Path(root)/'artifacts/notebook_15'/safe_id(run_id);out={'artifact':str(artifact)}
    for name in ['study_status','launch','heartbeat']:
        if(artifact/(name+'.json')).exists():out[name]=read(artifact/(name+'.json'))
    p=artifact/'visual_report/results.json'
    if p.exists():out['report']=read(p)
    return out
def validation_run(root,artifact):
    root,artifact=Path(root),Path(artifact);artifact.mkdir(parents=True,exist_ok=True)
    reports=[]
    for filename in ['15_Encoder_Runner_Validation.ipynb','15_Evaluation_Validation.ipynb']:
        path=root/'tests'/filename;m=types.ModuleType('fmm15_validation_'+filename.split('.')[0]);m.__file__=str(path);TaggedNotebookLoader(path).exec_module(m)
        reports.append(m.run_validation(root,artifact/filename.split('.')[0]))
    if any(x.get('status')!='PASS'for x in reports):raise AssertionError('Validation failed')
    path=root/'tests/15_Encoder_Runner_Validation.ipynb';fixtures=types.ModuleType('fmm15_workflow_fixtures');fixtures.__file__=str(path);TaggedNotebookLoader(path).exec_module(fixtures)
    fixture_plan=fixtures.fixture_plan(root,run_id=artifact.name,stage_id='integration',seed=20261791)
    fixture_plan['seeds']=[20261791,20261792,20261793,20261794];fixture_plan['registered_seeds']=fixture_plan['seeds']
    fixture_plan['arms']=[arm for seed in fixture_plan['seeds']for arm in fixtures.fixture_plan(root,run_id=artifact.name,stage_id='integration',seed=seed)['arms']]
    fixture_plan['quality_gate']=study_configuration()['quality_gate']
    fixture_plan['probe_convex']={'num_classes':3,'l2':.1,'max_steps':300,'gradient_inf_tolerance':1e-5,'relative_objective_tolerance':1e-9,'stable_checks':3}
    fixture_bundle=fixtures.fixture_bundle(root,artifact/'fixture_data');runner,evaluator=load(root,'runner'),load(root,'evaluation')
    configure(fixture_plan['arms'][0]['config']);immutable(artifact/'plan.json',fixture_plan)
    runner.prepare_stage(root,artifact,fixture_plan,fixture_bundle)
    resume=runner.verify_stage_resume(root,artifact,fixture_plan,fixture_bundle,time.time()+240)
    if resume['status']!='PASS':raise AssertionError('Integratedcheckpointpreflightfailed')
    training=runner.train_stage(root,artifact,fixture_plan,fixture_bundle,time.time()+240);atomic_json(artifact/'training_result.json',training)
    evaluation=evaluator.evaluate_stage(root,artifact,fixture_plan,fixture_bundle,training,time.time()+240)
    if evaluation['status']!='COMPLETE':raise AssertionError('Actualtinyinitial/finalevaluationincomplete')
    evaluation['fixture_source']='SYNTHETIC_SOFTWARE_VALIDATION_ONLY; NOT_RESEARCH_RESULTS';atomic_json(artifact/'stage_evaluation.json',evaluation)
    result=make_report(root,artifact,fixture_plan,evaluation)
    result['fixture_source']=evaluation['fixture_source'];atomic_json(artifact/'visual_report/results.json',result)
    handoff=read(artifact/'handoff_15_to_16.json');handoff['status']='VALIDATION_ONLY';handoff['quality_pass']=False;handoff['fixture_source']=result['fixture_source'];atomic_json(artifact/'handoff_15_to_16.json',handoff)
    receipt={'status':'PASS','profile':'validation','fixture_source':result['fixture_source'],'actual_tests':reports,
      'screening_logic':result['screening'],'figures':result['figures'],'no_live_experiment_launched':True}
    atomic_json(artifact/'validation_receipt.json',receipt);return receipt
